## Performance

I'm going to run crossmatches between ZTF and Gaia and have a look at the metrics.

In [ ]:
import lsdb
import ray

from dask.distributed import Client
from lsdb import ConeSearch
from ray.util.dask import enable_dask_on_ray


def run_crossmatch(radius_deg):
    """Run crossmatch between ZTF and Gaia catalogs for a specified radius"""
    ztf = lsdb.open_catalog(
        "/mnt/data/hats/catalogs/ztf_dr22",
        columns=["objectid", "objra", "objdec"],
        search_filter=ConeSearch(ra=270, dec=20, radius_arcsec=radius_deg*3600),
    )
    gaia = lsdb.open_catalog(
        "/mnt/data/hats/catalogs/v06/gaia_dr3",
        columns=["source_id", "ra", "dec"],
        search_filter=ConeSearch(ra=270, dec=20, radius_arcsec=radius_deg*3600),
    )
    return ztf.crossmatch(gaia)

def crossmatch_ray(catalog):
    """Using the dask-on-ray scheduler"""
    ray.init(num_cpus=4)
    with enable_dask_on_ray():
        catalog.compute()
    ray.shutdown()

def crossmatch_dask(catalog):
    """Using the native Dask scheduler"""
    with Client(n_workers=4, memory_limit=None):
        catalog.compute()

Let's compare the crossmatch performance with the increasing radius size:

In [ ]:
import human_readable
import sys
import time

def run_with_deg(radius_deg):
    catalog = run_crossmatch(radius_deg)

    graph = catalog._ddf.dask
    n_partitions = len(catalog.get_healpix_pixels())

    # Measure graph size in memory
    graph_size = 0
    for key in graph.keys():
        graph_size += sys.getsizeof(graph[key])
    graph_size = human_readable.file_size(graph_size, binary=True)
    
    print(f"Radius (deg): {radius_deg}")
    print(f"Num xmatch partitions: {n_partitions}")
    print(f"Graph len: {len(graph)}")
    print(f"Graph mem size: {graph_size}")

    print("Running with native Dask...")
    start_time = time.time()
    crossmatch_dask(catalog)
    print(f"Native Dask time: {time.time() - start_time}s")

    print("Running with Dask on Ray...")
    start_time = time.time()
    crossmatch_ray(catalog)
    print(f"Dask on Ray time: {time.time() - start_time}s")

In [ ]:
run_with_deg(1)

In [ ]:
run_with_deg(5)

In [ ]:
run_with_deg(10)

### Takeaways

The major difference between Dask and Ray is how memory is handled:

|  | Dask `memory_limit` | Ray `_memory` |
|---|---|---|
| Scope | Per worker | Total node budget |
| Enforced at the OS-level?| Yes | No |
| OOM behavior | Spill to disk / restart worker | No action |
| Used for scheduling? | Yes | Yes |

We can set memory requirements for tasks, but again, they are only enforced in scheduling:

```python
from ray.util.dask import RayDaskCallback

def annotate_memory(key, object_refs):
    return {"memory": 2 * 1024**3}

with RayDaskCallback(ray_pretask=annotate_memory):
    result = dask_dataframe.compute()
```

Using dask-on-ray in shared environments (e.g. arnor) is therefore, **not advisable**.